# Clase 3 — Few-shot y zero-shot prompting

Una de las formas más efectivas de mejorar la calidad de una respuesta es **mostrarle al modelo ejemplos** de lo que esperás. Eso es few-shot prompting. En esta clase vamos a entender cuándo usarlo, cómo construir buenos ejemplos y qué diferencia hace la cantidad de ejemplos.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Configuración del entorno |
| 2 | Zero-shot: el punto de partida |
| 3 | One-shot: un ejemplo cambia todo |
| 4 | Few-shot: cuántos ejemplos son suficientes |
| 5 | Cómo elegir buenos ejemplos |
| 6 | Actividad: diseñar una estrategia few-shot |

---
## 1. Configuración del entorno

**Si es tu primera vez en este curso:**

1. Obtené tu API key gratuita en [aistudio.google.com](https://aistudio.google.com) → **Get API key**.
2. Guardala en un archivo `.env` en la carpeta del proyecto:
   ```bash
   echo 'GEMINI_API_KEY=TU_CLAVE_AQUI' >> .env
   ```
3. Si preferís no crear el archivo, la celda siguiente te pide la clave de forma interactiva.

Si ya configuraste el entorno en clases anteriores, solo verificá que `BACKEND` esté correcto y ejecutá las celdas de setup.

In [ ]:
import os
import getpass

BACKEND = "gemini"
GEMINI_MODEL = "gemini-2.5-flash-lite"

if BACKEND == "gemini":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = getpass.getpass("Ingresá tu API key de Gemini: ")

print(f"Backend: {BACKEND}")

In [ ]:
if BACKEND == "gemini":
    from google import genai
    from google.genai import types
    _cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
elif BACKEND == "local":
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    ruta_modelo = hf_hub_download(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        filename="qwen2.5-0.5b-instruct-q4_k_m.gguf"
    )
    _llm_local = Llama(model_path=ruta_modelo, n_ctx=2048, n_gpu_layers=0, verbose=False)


def llamar_llm(prompt, system_prompt="Sos un asistente útil y conciso.", temperature=0.7, max_tokens=200):
    if BACKEND == "gemini":
        r = _cliente_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )
        return r.text.strip()
    elif BACKEND == "local":
        r = _llm_local.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r["choices"][0]["message"]["content"].strip()


print(llamar_llm("Respondé solo: 'Entorno listo.'", max_tokens=10))

---
## 2. Zero-shot: el punto de partida

**Zero-shot** significa darle al modelo una instrucción sin ningún ejemplo. Es la forma más simple de prompting y funciona bien para tareas genéricas. El problema aparece cuando la tarea tiene un formato específico o un estilo particular que el modelo no puede adivinar.

| Modo | Ejemplos en el prompt | Cuándo usar |
|---|---|---|
| **Zero-shot** | 0 | Tareas genéricas, primeros intentos, cuando no tenés ejemplos buenos |
| **One-shot** | 1 | Cuando tenés un ejemplo claro y representativo |
| **Few-shot** | 2–5 | Cuando necesitás consistencia de formato o estilo específico |

In [ ]:
# ─── Tarea de referencia ─────────────────────────────────────────────────────
# Vamos a clasificar el sentimiento de comentarios de clientes.
# Queremos: una sola palabra (Positivo / Negativo / Neutro) + una frase de justificación.

comentarios_prueba = [
    "El producto llegó antes de lo esperado y funciona perfecto.",
    "Tardó tres semanas y el empaque estaba roto.",
    "Lo usé una vez, sirve para lo que dice."
]

# ─── Zero-shot ────────────────────────────────────────────────────────────────
def clasificar_zero_shot(comentario):
    prompt = f"Clasificá el sentimiento de este comentario de cliente: '{comentario}'"
    return llamar_llm(prompt, max_tokens=60)


print("=== ZERO-SHOT ===")
for c in comentarios_prueba:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_zero_shot(c)}")
    print()

> 💡 **Observá:** con zero-shot el modelo entiende la tarea, pero el formato de la respuesta varía — a veces da una palabra, a veces un párrafo. Eso dificulta procesar los resultados automáticamente.

---
## 3. One-shot: un ejemplo cambia todo

Con un solo ejemplo bien elegido, el modelo entiende no solo *qué* hacer sino *cómo* quiero que lo presente.

In [ ]:
# ─── One-shot ─────────────────────────────────────────────────────────────────
# Ahora incluimos un ejemplo explícito de cómo debe verse la respuesta.

def clasificar_one_shot(comentario):
    prompt = """Clasificá el sentimiento de comentarios de clientes.
El formato de respuesta es: Sentimiento: <Positivo|Negativo|Neutro> | Razón: <una frase corta>

Ejemplo:
Comentario: "El servicio fue rápido pero el producto tenía un defecto menor."
Sentimiento: Neutro | Razón: mezcla aspectos positivos y negativos sin dominante claro.

Ahora clasificá este:
Comentario: "{comentario}"""

    return llamar_llm(prompt.format(comentario=comentario), max_tokens=60)


print("=== ONE-SHOT ===")
for c in comentarios_prueba:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_one_shot(c)}")
    print()

---
## 4. Few-shot: cuántos ejemplos son suficientes

Agregar más ejemplos generalmente mejora la consistencia, pero llega un punto de rendimientos decrecientes. Después de 3–5 ejemplos bien elegidos, agregar más raramente cambia el resultado.

In [ ]:
# ─── Few-shot con 3 ejemplos ──────────────────────────────────────────────────

ejemplos_few_shot = [
    ("La atención fue excelente y me solucionaron el problema en minutos.",
     "Sentimiento: Positivo | Razón: resolución rápida y efectiva."),
    ("El producto no coincide con las fotos del sitio.",
     "Sentimiento: Negativo | Razón: discrepancia entre lo mostrado y lo recibido."),
    ("Funciona como indica el manual, sin más ni menos.",
     "Sentimiento: Neutro | Razón: cumple expectativas básicas sin destacar."),
]

def clasificar_few_shot(comentario, ejemplos):
    # Construimos el bloque de ejemplos dinámicamente
    bloque_ejemplos = "\n".join(
        f'Comentario: "{inp}"\n{out}'
        for inp, out in ejemplos
    )

    prompt = f"""Clasificá el sentimiento de comentarios de clientes.
Formato: Sentimiento: <Positivo|Negativo|Neutro> | Razón: <una frase corta>

{bloque_ejemplos}

Ahora clasificá este:
Comentario: "{comentario}"""

    return llamar_llm(prompt, max_tokens=60)


print("=== FEW-SHOT (3 ejemplos) ===")
for c in comentarios_prueba:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_few_shot(c, ejemplos_few_shot)}")
    print()

In [ ]:
# ─── Comparación de consistencia por cantidad de ejemplos ─────────────────────
# Ejecutamos el mismo comentario con 0, 1 y 3 ejemplos y comparamos.

comentario_test = "El envío tardó más de lo prometido pero el producto es de buena calidad."

print("Comentario:", comentario_test)
print()
print("Zero-shot: ", clasificar_zero_shot(comentario_test))
print("One-shot:  ", clasificar_one_shot(comentario_test))
print("Few-shot:  ", clasificar_few_shot(comentario_test, ejemplos_few_shot))

---
## 5. Cómo elegir buenos ejemplos

No todos los ejemplos son igual de útiles. Estos criterios ayudan a elegirlos:

| Criterio | Qué significa |
|---|---|
| **Representatividad** | El ejemplo cubre un caso típico, no un caso raro |
| **Diversidad** | Si tenés varios ejemplos, que no sean todos del mismo tipo |
| **Claridad** | El par input-output es inequívoco; no hay ambigüedad en por qué la respuesta es esa |
| **Longitud similar** | Los ejemplos deberían tener longitud parecida al input real |
| **Sin contradicciones** | Dos ejemplos no deben sugerir respuestas distintas para inputs similares |

> 💡 Un ejemplo mal elegido puede confundir más que ayudar. Antes de agregar ejemplos, verificá que cada uno cumpla estos criterios.

In [ ]:
# ─── Efecto de un ejemplo de mala calidad ─────────────────────────────────────
# Incluimos un ejemplo contradictorio para ver cómo afecta al modelo.

ejemplos_malos = [
    # Ejemplo contradictorio: input claramente positivo, output incorrecto
    ("Excelente atención, quedé muy satisfecho.",
     "Sentimiento: Negativo | Razón: el cliente mencionó un problema."),
]

print("=== FEW-SHOT CON EJEMPLO CONTRADICTORIO ===")
for c in comentarios_prueba[:2]:   # solo los primeros dos para ahorrar tokens
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_few_shot(c, ejemplos_malos)}")
    print()

print("Comparar con few-shot limpio:")
for c in comentarios_prueba[:2]:
    print(f"Comentario: {c}")
    print(f"Respuesta:  {clasificar_few_shot(c, ejemplos_few_shot)}")
    print()

---
## 6. Actividad: diseñar una estrategia few-shot

Elegí una tarea de tu trabajo o área (clasificación, extracción de datos, generación de texto con formato específico). Construí un set de 3 ejemplos few-shot y probá la diferencia respecto al zero-shot.

In [ ]:
# TODO: Definí tu tarea y construí los ejemplos
# Reemplazá los "..." con tu contenido

# Descripción de la tarea:
# ...

mis_ejemplos = [
    # ("input de ejemplo 1", "output esperado 1"),
    # ("input de ejemplo 2", "output esperado 2"),
    # ("input de ejemplo 3", "output esperado 3"),
]

mis_inputs_prueba = [
    # "input 1 para probar",
    # "input 2 para probar",
]

# ─── Zero-shot de tu tarea ────────────────────────────────────────────────────
instruccion_zero = """..."""   # Describí la tarea en una oración

for inp in mis_inputs_prueba:
    print(f"Input: {inp}")
    print(f"Zero-shot: {llamar_llm(instruccion_zero + ' ' + inp, max_tokens=80)}")
    print(f"Few-shot:  {clasificar_few_shot(inp, mis_ejemplos)}")
    print()

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es la actividad de la sección 6 completa: tarea definida, 3 ejemplos few-shot y comparación de resultados con zero-shot.

**Para la próxima clase:** chain-of-thought — cómo hacer que el modelo muestre su razonamiento para mejorar la precisión en tareas complejas.